# Name Hierarchy Levels (Mid & High)

Assigns globally unique, publication-ready names to mid-level and high-level
topic groups for **both** corpora (papers and iGEM teams).

For each group the LLM sees the low-level sub-topics (name + description)
that belong to it, then returns a single name per group using OpenAI
function calling.

**Prerequisites** — run these notebooks first:
- `get_papers_topic_hierarchy.ipynb`
- `get_teams_topic_hierarchy.ipynb`

**Inputs:**
- `assets/topic_models/{papers,teams}_topic_names.txt` — low-level topic names & descriptions
- `assets/reports/{papers,teams}_topic_name_hierarchy.tsv` — low → mid → high mapping
- `prompts_hierarchy.yaml` — LLM system prompt template
- `../03-topic_names/openai.key` — API key

**Outputs:**
- Updates `assets/reports/{papers,teams}_topic_name_hierarchy.tsv` with `mid_name` and `high_name` columns

In [10]:
import json
from pathlib import Path

import pandas as pd
import yaml
from openai import OpenAI

ROOT_DIR = Path.cwd().parent
ASSETS_DIR = ROOT_DIR / "assets"
MODELS_DIR = ASSETS_DIR / "topic_models"
REPORTS_DIR = ASSETS_DIR / "reports"

MODEL = "gpt-4.1-nano"

# Load prompts
with open("prompts_hierarchy.yaml") as f:
    prompts = yaml.safe_load(f)

THEME = prompts["theme"]
THEME_DESCRIPTION = prompts["theme_description"]
SYSTEM_TEMPLATE = prompts["hierarchy_naming"]["system"]
SYSTEM_PROMPT = SYSTEM_TEMPLATE.format(theme=THEME, theme_description=THEME_DESCRIPTION)

# OpenAI client
api_key = (Path.cwd().parent / "03-topic_names" / "openai.key").read_text().strip()
client = OpenAI(api_key=api_key)

print("OpenAI client ready")
print(f"Model: {MODEL}")

OpenAI client ready
Model: gpt-4.1-nano


In [11]:
# ── Helper functions ──────────────────────────────────────────────────────────

def build_rename_tool(n_groups: int) -> dict:
    """OpenAI function-calling tool schema for naming hierarchy groups."""
    return {
        "type": "function",
        "function": {
            "name": "name_groups",
            "description": (
                f"Assign a unique, concise, publication-ready name to each of "
                f"the {n_groups} topic groups. Every name must be distinct."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "groups": {
                        "type": "array",
                        "description": f"Exactly {n_groups} named groups.",
                        "items": {
                            "type": "object",
                            "properties": {
                                "group_id": {
                                    "type": "integer",
                                    "description": "The numeric group ID.",
                                },
                                "name": {
                                    "type": "string",
                                    "description": (
                                        "A short, unique name for this group. "
                                        "Must be distinct from every other group name."
                                    ),
                                },
                            },
                            "required": ["group_id", "name"],
                        },
                    },
                },
                "required": ["groups"],
            },
        },
    }


def build_user_message(
    hierarchy_df: pd.DataFrame,
    topic_names: pd.DataFrame,
    level_col: str,
) -> str:
    """Format groups for the LLM: each group shows its sub-topic names & descriptions."""
    # Merge low-level names/descriptions into the hierarchy
    merged = hierarchy_df.merge(
        topic_names[["low", "global_name", "description"]],
        on="low",
        how="left",
    )
    lines = []
    for gid, grp in merged.groupby(level_col):
        if int(gid) < 0:
            continue
        lines.append(f"Group {int(gid)}:")
        for _, row in grp.iterrows():
            name = row.get("global_name", row.get("name", "unknown"))
            desc = row.get("description", "")
            lines.append(f"  - {name}: {desc}")
        lines.append("")
    return "\n".join(lines)


def name_hierarchy_level(
    hierarchy_df: pd.DataFrame,
    topic_names: pd.DataFrame,
    level_col: str,
    label: str,
) -> dict[int, str]:
    """Call OpenAI to name all groups at one hierarchy level.

    Returns {group_id: name}.
    """
    n_groups = hierarchy_df[hierarchy_df[level_col] >= 0][level_col].nunique()
    print(f"  Naming {n_groups} {label} groups via {MODEL} …")

    tool = build_rename_tool(n_groups)
    user_msg = build_user_message(hierarchy_df, topic_names, level_col)

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        tools=[tool],
        tool_choice={"type": "function", "function": {"name": "name_groups"}},
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
    )

    call = response.choices[0].message.tool_calls[0]
    args = json.loads(call.function.arguments)
    mapping = {item["group_id"]: item["name"] for item in args["groups"]}

    # Verify completeness and uniqueness
    expected_ids = set(hierarchy_df[hierarchy_df[level_col] >= 0][level_col].unique())
    missing = expected_ids - set(mapping.keys())
    if missing:
        print(f"  WARNING: {len(missing)} group(s) did not receive a name: {missing}")
    dupes = len(mapping.values()) - len(set(mapping.values()))
    if dupes:
        print(f"  WARNING: {dupes} duplicate name(s) found")

    print(f"  ✓ {len(mapping)} {label} names assigned")
    return mapping

print("Helper functions defined ✓")

Helper functions defined ✓


## Papers hierarchy naming

In [12]:
# Load papers data
papers_topic_names = pd.read_csv(MODELS_DIR / "papers_topic_names.txt", sep="\t")
papers_topic_names = papers_topic_names.rename(columns={"topic": "low"})

papers_hierarchy = pd.read_csv(REPORTS_DIR / "papers_topic_name_hierarchy.tsv", sep="\t")

print(f"Papers: {len(papers_topic_names)} low-level topics")
print(f"  mid groups : {papers_hierarchy[papers_hierarchy['mid'] >= 0]['mid'].nunique()}")
print(f"  high groups: {papers_hierarchy[papers_hierarchy['high'] >= 0]['high'].nunique()}")

Papers: 263 low-level topics
  mid groups : 44
  high groups: 4


In [13]:
# Name papers mid-level groups
print("Papers — mid level:")
papers_mid_names = name_hierarchy_level(
    papers_hierarchy, papers_topic_names,
    level_col="mid", label="mid",
)

# Name papers high-level groups
print("Papers — high level:")
papers_high_names = name_hierarchy_level(
    papers_hierarchy, papers_topic_names,
    level_col="high", label="high",
)

Papers — mid level:
  Naming 44 mid groups via gpt-4.1-nano …
  ✓ 44 mid names assigned
Papers — high level:
  Naming 4 high groups via gpt-4.1-nano …
  ✓ 4 high names assigned


In [14]:
# Add mid_name and high_name columns and save
papers_hierarchy["mid_name"] = papers_hierarchy["mid"].map(papers_mid_names)
papers_hierarchy["high_name"] = papers_hierarchy["high"].map(papers_high_names)

papers_out = REPORTS_DIR / "papers_topic_name_hierarchy.tsv"
papers_hierarchy.to_csv(papers_out, sep="\t", index=False)

print(f"Saved → {papers_out.name}")
papers_hierarchy.head(10)

Saved → papers_topic_name_hierarchy.tsv


,global_name,low,mid,high,mid_name,high_name
0,Genetic Circuit Control,0,0,0,Genetic Circuit Engineering and Control,Foundations and Applications of Synthetic Biol...
1,Synthetic Nucleic Acid Design,1,1,1,Nucleic Acid Structural and Functional Enginee...,"Synthetic Nucleic Acids, Gene Regulation, and ..."
2,Plant Biosynthetic Pathways,2,2,0,Metabolic Pathway and Microbial Engineering fo...,Foundations and Applications of Synthetic Biol...
3,Synthetic CpG ODNs in Immunity,3,3,1,Synthetic Immunomodulation and Cancer Therapy,"Synthetic Nucleic Acids, Gene Regulation, and ..."
4,Ethics and Governance in Synthetic Biology,4,4,2,"Societal, Ethical, and Educational Aspects of ...","Societal, Ethical, and Governance Aspects of S..."
5,Genetic Circuits and Logic Devices,5,0,0,Genetic Circuit Engineering and Control,Foundations and Applications of Synthetic Biol...
6,Oligonucleotide Modification and Surface Funct...,6,5,1,Synthetic Nucleic Acid Tools and DNA Data Storage,"Synthetic Nucleic Acids, Gene Regulation, and ..."
7,Synthetic Genome Construction,7,6,0,Synthetic Genome Design and Bacterial Genome E...,Foundations and Applications of Synthetic Biol...
8,Yeast Gene Regulation and Protein Function,8,7,1,Gene Regulation and Protein Function in Yeast,"Synthetic Nucleic Acids, Gene Regulation, and ..."
9,Synthetic Biological Systems and Circuits,9,0,0,Genetic Circuit Engineering and Control,Foundations and Applications of Synthetic Biol...


## Teams hierarchy naming

In [15]:
# Load teams data
teams_topic_names = pd.read_csv(MODELS_DIR / "teams_topic_names.txt", sep="\t")
teams_topic_names = teams_topic_names.rename(columns={"topic": "low"})

teams_hierarchy = pd.read_csv(REPORTS_DIR / "teams_topic_name_hierarchy.tsv", sep="\t")

print(f"Teams: {len(teams_topic_names)} low-level topics")
print(f"  mid groups : {teams_hierarchy[teams_hierarchy['mid'] >= 0]['mid'].nunique()}")
print(f"  high groups: {teams_hierarchy[teams_hierarchy['high'] >= 0]['high'].nunique()}")

Teams: 161 low-level topics
  mid groups : 25
  high groups: 10


In [16]:
# Name teams mid-level groups
print("Teams — mid level:")
teams_mid_names = name_hierarchy_level(
    teams_hierarchy, teams_topic_names,
    level_col="mid", label="mid",
)

# Name teams high-level groups
print("Teams — high level:")
teams_high_names = name_hierarchy_level(
    teams_hierarchy, teams_topic_names,
    level_col="high", label="high",
)

Teams — mid level:
  Naming 25 mid groups via gpt-4.1-nano …
  ✓ 25 mid names assigned
Teams — high level:
  Naming 10 high groups via gpt-4.1-nano …
  ✓ 10 high names assigned


In [17]:
# Add mid_name and high_name columns and save
teams_hierarchy["mid_name"] = teams_hierarchy["mid"].map(teams_mid_names)
teams_hierarchy["high_name"] = teams_hierarchy["high"].map(teams_high_names)

teams_out = REPORTS_DIR / "teams_topic_name_hierarchy.tsv"
teams_hierarchy.to_csv(teams_out, sep="\t", index=False)

print(f"Saved → {teams_out.name}")
teams_hierarchy.head(10)

Saved → teams_topic_name_hierarchy.tsv


,global_name,low,mid,high,mid_name,high_name
0,Synthetic Diagnostics & Biosensing Technologies,0,0,0,Next-Generation Diagnostics and Biosensing,Synthetic Diagnostics and Biosensing Technologies
1,Light-Controlled Biological Systems,1,1,1,Optogenetics and Genetic Control Systems,Optogenetic Control and Genetic Circuitry
2,Synthetic Microbial Detection & Biofilm Manage...,2,2,2,Synthetic Biology for Environmental and Public...,Synthetic Biology for Environmental and Medica...
3,Synthetic Plastic Degradation & Recycling,3,3,3,Biodegradation and Circular Economy Solutions,Biodegradation and Sustainable Waste Management
4,Synthetic Therapeutics for Cancer & Diseases,4,4,4,Synthetic Biology in Cancer and Therapeutics,Synthetic Therapeutics and Disease Intervention
5,Microbial Diagnostic & Therapeutic Platforms,5,5,5,Microbial Diagnostics and Therapeutic Microbiomes,Microbial Diagnostics and Health Monitoring
6,Synthetic Blood & Metabolic Biosystems,6,0,0,Next-Generation Diagnostics and Biosensing,Synthetic Diagnostics and Biosensing Technologies
7,Genetic Control & Diagnostic Circuits,7,1,1,Optogenetics and Genetic Control Systems,Optogenetic Control and Genetic Circuitry
8,Synthetic Design & Automation Tools,8,6,1,Synthetic Biology Tools and Automation,Optogenetic Control and Genetic Circuitry
9,Genetic & Molecular Control Technologies,9,1,1,Optogenetics and Genetic Control Systems,Optogenetic Control and Genetic Circuitry


In [18]:
# Summary
print("=== Naming complete ===")
for label, hier in [("Papers", papers_hierarchy), ("Teams", teams_hierarchy)]:
    n_mid = hier["mid_name"].notna().sum()
    n_high = hier["high_name"].notna().sum()
    mid_unique = hier["mid_name"].dropna().nunique()
    high_unique = hier["high_name"].dropna().nunique()
    print(f"  {label}: {n_mid} topics with mid_name ({mid_unique} unique), "
          f"{n_high} topics with high_name ({high_unique} unique)")

=== Naming complete ===
  Papers: 263 topics with mid_name (44 unique), 263 topics with high_name (4 unique)
  Teams: 161 topics with mid_name (25 unique), 161 topics with high_name (10 unique)
